# 15 — Live adjustment engine for WC2026 paper picks

After the tournament starts, one static forecast is not enough. Probabilities need to be updated regularly.

This notebook answers:
- is the tracked pick still alive?
- did the market move toward or against the model?
- is it better to track now or wait for a better line?
- did the edge increase or disappear?
- which matches require manual review because of lineups, injuries, motivation, weather, or group-table context?

This notebook builds an adjustment layer on top of:
1. base probabilities from notebook 12;
2. fresh odds from notebook 04;
3. historical calibration from notebook 11;
4. line movement, closing-line-value proxy, and time-to-kickoff logic.

Important notes:
- `real_money_stake_usd = 0`;
- output is a paper-monitoring table;
- if the odds ledger contains multiple notebook 04 runs, this notebook uses line movement over time.


In [ ]:
# Cell 1. Configuration.

MIN_EDGE = 0.05
MIN_EV = 0.00
MIN_ODDS = 1.80
MAX_ODDS = 6.00

LINE_MOVE_STRONG = 0.02
LINE_MOVE_WARNING = -0.02

WAIT_EDGE_LOW = 0.06
FAR_FROM_KICKOFF_HOURS = 24

CLOSE_TO_KICKOFF_HOURS = 6
LATE_MARKET_SHRINK = 0.20

REAL_MONEY_STAKE_USD = 0.0


In [ ]:
# Cell 2. Imports and helpers.

from pathlib import Path
import json
import zipfile

import numpy as np
import pandas as pd

DATA_DIR = Path("data")
PROCESSED_DIR = DATA_DIR / "processed"

OUT_DIR = PROCESSED_DIR / "15_live_adjustment_engine"
OUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP_UTC = pd.Timestamp.utcnow()

def normalize_probs(df, cols):
    arr = df[cols].to_numpy(dtype=float)
    arr = np.clip(arr, 1e-6, 1.0)
    arr = arr / arr.sum(axis=1, keepdims=True)

    out = df.copy()

    for i, col in enumerate(cols):
        out[col] = arr[:, i]

    return out

def save_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "w", encoding="utf-8") as f:
        json.dump(
            payload,
            f,
            indent=2,
            ensure_ascii=False,
            default=str,
        )

    return path

print("Setup OK.")


In [ ]:
# Cell 3. Load base predictions and odds ledger.

base_predictions_path = (
    PROCESSED_DIR
    / "12_current_wc2026_paper_picks"
    / "current_match_predictions.csv"
)

odds_ledger_path = (
    PROCESSED_DIR
    / "current_wc2026_odds_ledger"
    / "wc2026_current_odds_ledger.csv"
)

if not base_predictions_path.exists():
    raise FileNotFoundError(
        "Run 12_current_wc2026_paper_picks.ipynb first."
    )

if not odds_ledger_path.exists():
    raise FileNotFoundError(
        "Run 04_current_wc2026_odds_ledger.ipynb first."
    )

base = pd.read_csv(base_predictions_path)
ledger = pd.read_csv(odds_ledger_path)

base["commence_time"] = pd.to_datetime(
    base["commence_time"],
    utc=True,
    errors="coerce",
)

ledger["commence_time"] = pd.to_datetime(
    ledger["commence_time"],
    utc=True,
    errors="coerce",
)

if "run_timestamp_utc" in ledger.columns:
    ledger["run_timestamp_utc"] = pd.to_datetime(
        ledger["run_timestamp_utc"],
        utc=True,
        errors="coerce",
    )
else:
    ledger["run_timestamp_utc"] = RUN_TIMESTAMP_UTC

base = base.dropna(subset=["event_id", "commence_time"]).copy()
ledger = ledger.dropna(subset=["event_id", "commence_time"]).copy()

base["event_id"] = base["event_id"].astype(str)
ledger["event_id"] = ledger["event_id"].astype(str)

print("base predictions:", base.shape)
print("odds ledger rows:", ledger.shape)

display(base.head())
display(ledger.head())


In [ ]:
# Cell 4. Build latest odds snapshot and opening/reference snapshot.

latest = (
    ledger
    .sort_values(["event_id", "run_timestamp_utc"])
    .groupby("event_id", as_index=False)
    .tail(1)
    .copy()
)

opening = (
    ledger
    .sort_values(["event_id", "run_timestamp_utc"])
    .groupby("event_id", as_index=False)
    .head(1)
    .copy()
)

latest_cols = [
    "event_id",
    "run_timestamp_utc",
    "market_p_away_devig",
    "market_p_draw_devig",
    "market_p_home_devig",
    "best_away_odds",
    "best_draw_odds",
    "best_home_odds",
    "avg_away_odds",
    "avg_draw_odds",
    "avg_home_odds",
    "n_bookmakers",
    "market_margin_avg",
]

opening_cols = [
    "event_id",
    "run_timestamp_utc",
    "market_p_away_devig",
    "market_p_draw_devig",
    "market_p_home_devig",
    "best_away_odds",
    "best_draw_odds",
    "best_home_odds",
]

latest_small = latest[
    [c for c in latest_cols if c in latest.columns]
].copy()

opening_small = opening[
    [c for c in opening_cols if c in opening.columns]
].copy()

latest_small = latest_small.rename(columns={
    "run_timestamp_utc": "latest_odds_timestamp",
    "market_p_away_devig": "latest_market_p_away",
    "market_p_draw_devig": "latest_market_p_draw",
    "market_p_home_devig": "latest_market_p_home",
    "best_away_odds": "latest_best_away_odds",
    "best_draw_odds": "latest_best_draw_odds",
    "best_home_odds": "latest_best_home_odds",
    "avg_away_odds": "latest_avg_away_odds",
    "avg_draw_odds": "latest_avg_draw_odds",
    "avg_home_odds": "latest_avg_home_odds",
    "n_bookmakers": "latest_n_bookmakers",
    "market_margin_avg": "latest_market_margin_avg",
})

opening_small = opening_small.rename(columns={
    "run_timestamp_utc": "opening_odds_timestamp",
    "market_p_away_devig": "opening_market_p_away",
    "market_p_draw_devig": "opening_market_p_draw",
    "market_p_home_devig": "opening_market_p_home",
    "best_away_odds": "opening_best_away_odds",
    "best_draw_odds": "opening_best_draw_odds",
    "best_home_odds": "opening_best_home_odds",
})

odds_state = latest_small.merge(
    opening_small,
    on="event_id",
    how="left",
)

print("latest odds:", latest_small.shape)
print("odds state:", odds_state.shape)

display(odds_state.head())


In [ ]:
# Cell 5. Merge base model with current market state.

keep_base_cols = [
    "event_id",
    "commence_time",
    "home_team",
    "away_team",
    "final_p_away",
    "final_p_draw",
    "final_p_home",
    "market_p_away",
    "market_p_draw",
    "market_p_home",
    "football_p_away",
    "football_p_draw",
    "football_p_home",
    "score_p_away",
    "score_p_draw",
    "score_p_home",
    "outcome_model_p_away",
    "outcome_model_p_draw",
    "outcome_model_p_home",
    "most_likely_home_score",
    "most_likely_away_score",
]

keep_base_cols = [c for c in keep_base_cols if c in base.columns]

live = base[keep_base_cols].merge(
    odds_state,
    on="event_id",
    how="left",
)

for outcome in ["away", "draw", "home"]:
    live[f"latest_market_p_{outcome}"] = live[
        f"latest_market_p_{outcome}"
    ].fillna(live[f"market_p_{outcome}"])

    live[f"opening_market_p_{outcome}"] = live[
        f"opening_market_p_{outcome}"
    ].fillna(live[f"market_p_{outcome}"])

    live[f"latest_best_{outcome}_odds"] = live[
        f"latest_best_{outcome}_odds"
    ].fillna(np.nan)

now = pd.Timestamp.utcnow()

live["hours_to_kickoff"] = (
    live["commence_time"] - now
).dt.total_seconds() / 3600.0

live = live.sort_values("commence_time").reset_index(drop=True)

display(live.head())


In [ ]:
# Cell 6. Market-aware probability adjustment.

# If kickoff is close, shrink model toward latest market.
# This indirectly reacts to lineups, injuries, and late information.

adjusted = live.copy()

for outcome in ["away", "draw", "home"]:
    adjusted[f"adjusted_p_{outcome}"] = adjusted[f"final_p_{outcome}"]

close_mask = adjusted["hours_to_kickoff"] <= CLOSE_TO_KICKOFF_HOURS

for outcome in ["away", "draw", "home"]:
    adjusted.loc[close_mask, f"adjusted_p_{outcome}"] = (
        (1.0 - LATE_MARKET_SHRINK)
        * adjusted.loc[close_mask, f"final_p_{outcome}"]
        + LATE_MARKET_SHRINK
        * adjusted.loc[close_mask, f"latest_market_p_{outcome}"]
    )

adjusted = normalize_probs(
    adjusted,
    [
        "adjusted_p_away",
        "adjusted_p_draw",
        "adjusted_p_home",
    ],
)

pred_idx = adjusted[[
    "adjusted_p_away",
    "adjusted_p_draw",
    "adjusted_p_home",
]].to_numpy().argmax(axis=1)

adjusted["adjusted_predicted_outcome"] = np.array(
    ["away", "draw", "home"],
    dtype=object,
)[pred_idx]

display(adjusted[[
    "commence_time",
    "home_team",
    "away_team",
    "hours_to_kickoff",
    "final_p_home",
    "final_p_draw",
    "final_p_away",
    "adjusted_p_home",
    "adjusted_p_draw",
    "adjusted_p_away",
    "adjusted_predicted_outcome",
]].head(30))


In [ ]:
# Cell 7. Build adjusted selection table.

rows = []

for _, r in adjusted.iterrows():
    for selection in ["away", "draw", "home"]:
        model_p = r[f"adjusted_p_{selection}"]
        latest_market_p = r[f"latest_market_p_{selection}"]
        opening_market_p = r[f"opening_market_p_{selection}"]
        odds = r.get(f"latest_best_{selection}_odds", np.nan)

        if pd.isna(odds):
            continue

        edge = model_p - latest_market_p
        ev = model_p * odds - 1.0

        line_move = latest_market_p - opening_market_p

        opening_odds = r.get(f"opening_best_{selection}_odds", np.nan)

        if (
            not pd.isna(opening_odds)
            and not pd.isna(odds)
            and odds > 0
        ):
            odds_move = opening_odds / odds - 1.0
        else:
            odds_move = np.nan

        passes = (
            edge >= MIN_EDGE
            and ev >= MIN_EV
            and odds >= MIN_ODDS
            and odds <= MAX_ODDS
        )

        status = "skip"

        if passes:
            status = "paper_take"

        if passes and r["hours_to_kickoff"] > FAR_FROM_KICKOFF_HOURS:
            if edge < WAIT_EDGE_LOW:
                status = "paper_wait"

        if passes and line_move <= LINE_MOVE_WARNING:
            status = "paper_warning_market_against"

        if passes and line_move >= LINE_MOVE_STRONG:
            status = "paper_take_market_confirming"

        if (
            "most_likely_home_score" in r
            and not pd.isna(r["most_likely_home_score"])
        ):
            likely_score = (
                f"{int(r['most_likely_home_score'])}-"
                f"{int(r['most_likely_away_score'])}"
            )
        else:
            likely_score = ""

        rows.append({
            "run_timestamp_utc": RUN_TIMESTAMP_UTC,
            "paper_only": True,
            "real_money_stake_usd": REAL_MONEY_STAKE_USD,
            "event_id": r["event_id"],
            "commence_time": r["commence_time"],
            "hours_to_kickoff": r["hours_to_kickoff"],
            "home_team": r["home_team"],
            "away_team": r["away_team"],
            "selection": selection,
            "latest_decimal_odds": odds,
            "opening_decimal_odds": opening_odds,
            "adjusted_probability": model_p,
            "latest_market_probability": latest_market_p,
            "opening_market_probability": opening_market_p,
            "edge": edge,
            "expected_value": ev,
            "market_probability_move": line_move,
            "odds_move_clv_proxy": odds_move,
            "passes_strategy": passes,
            "live_status": status,
            "most_likely_score": likely_score,
        })

selection_table = pd.DataFrame(rows)

selection_table.to_csv(
    OUT_DIR / "live_adjusted_all_selections.csv",
    index=False,
)

display(selection_table.head(20))


In [ ]:
# Cell 8. Final live paper watchlist.

watchlist = selection_table[
    selection_table["passes_strategy"]
].copy()

watchlist = watchlist.sort_values(
    [
        "commence_time",
        "live_status",
        "expected_value",
        "edge",
    ],
    ascending=[True, True, False, False],
).reset_index(drop=True)

human_cols = [
    "commence_time",
    "hours_to_kickoff",
    "home_team",
    "away_team",
    "selection",
    "live_status",
    "latest_decimal_odds",
    "adjusted_probability",
    "latest_market_probability",
    "edge",
    "expected_value",
    "market_probability_move",
    "odds_move_clv_proxy",
    "most_likely_score",
    "real_money_stake_usd",
]

watchlist_human = watchlist[human_cols].copy()

watchlist_human["match"] = (
    watchlist_human["home_team"]
    + " vs "
    + watchlist_human["away_team"]
)

watchlist_human = watchlist_human[[
    "commence_time",
    "hours_to_kickoff",
    "match",
    "selection",
    "live_status",
    "latest_decimal_odds",
    "adjusted_probability",
    "latest_market_probability",
    "edge",
    "expected_value",
    "market_probability_move",
    "odds_move_clv_proxy",
    "most_likely_score",
    "real_money_stake_usd",
]]

watchlist.to_csv(
    OUT_DIR / "live_adjusted_paper_picks.csv",
    index=False,
)

watchlist_human.to_csv(
    OUT_DIR / "human_live_adjusted_watchlist.csv",
    index=False,
)

display(watchlist_human.head(50))
print("live picks:", len(watchlist_human))


In [ ]:
# Cell 9. Match-level dashboard.

match_dashboard = adjusted.copy()

match_dashboard["max_adjusted_probability"] = match_dashboard[[
    "adjusted_p_away",
    "adjusted_p_draw",
    "adjusted_p_home",
]].max(axis=1)

market_pred_idx = match_dashboard[[
    "latest_market_p_away",
    "latest_market_p_draw",
    "latest_market_p_home",
]].to_numpy().argmax(axis=1)

match_dashboard["market_favorite"] = np.array(
    ["away", "draw", "home"],
    dtype=object,
)[market_pred_idx]

match_dashboard["model_vs_market_disagree"] = (
    match_dashboard["adjusted_predicted_outcome"]
    != match_dashboard["market_favorite"]
)

match_dashboard = match_dashboard.sort_values(
    ["commence_time"],
).reset_index(drop=True)

match_dashboard.to_csv(
    OUT_DIR / "live_adjusted_match_dashboard.csv",
    index=False,
)

display(match_dashboard[[
    "commence_time",
    "home_team",
    "away_team",
    "hours_to_kickoff",
    "adjusted_predicted_outcome",
    "market_favorite",
    "model_vs_market_disagree",
    "adjusted_p_home",
    "adjusted_p_draw",
    "adjusted_p_away",
    "latest_market_p_home",
    "latest_market_p_draw",
    "latest_market_p_away",
]].head(40))


In [ ]:
# Cell 10. Save report zip.

summary = {
    "run_timestamp_utc": RUN_TIMESTAMP_UTC,
    "matches": int(len(adjusted)),
    "all_selection_rows": int(len(selection_table)),
    "live_paper_picks": int(len(watchlist_human)),
    "min_edge": MIN_EDGE,
    "min_ev": MIN_EV,
    "min_odds": MIN_ODDS,
    "max_odds": MAX_ODDS,
    "line_move_strong": LINE_MOVE_STRONG,
    "line_move_warning": LINE_MOVE_WARNING,
    "late_market_shrink": LATE_MARKET_SHRINK,
    "close_to_kickoff_hours": CLOSE_TO_KICKOFF_HOURS,
    "real_money_allowed": False,
}

save_json(
    OUT_DIR / "summary.json",
    summary,
)

zip_path = (
    PROCESSED_DIR
    / "15_live_adjustment_engine_report_bundle.zip"
)

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as zf:
    for file in OUT_DIR.rglob("*"):
        if file.is_file():
            zf.write(file, arcname=file.relative_to(OUT_DIR))

display(pd.DataFrame([summary]))

print("Created:", zip_path.resolve())
print("Report bundle created.")
